# 01 · Identify companies by headquarters country

**LSEG Workspace · CodeBook · Python**

**Objective.** Identify public primary equity records with headquarters in Spain, retrieve their reference identifiers, sort the results alphabetically, and export a CSV.

**Research distinction:** Headquarters in Spain does not mean that a multinational operates *only* in Spain, and the number of returned equity records is not necessarily the number of unique legal entities. Verify the `SCREEN()` formula in Workspace Screener / CodeCreator for your access and screening criteria.

## Step 1 · Import libraries and open a session

In [ ]:
import refinitiv.data as rd
import pandas as pd
from pathlib import Path

rd.open_session()

## Step 2 · Define the screening universe

In [ ]:
# ES is the ISO two-letter country code for Spain.
# Use Workspace Screener to verify or adjust the screening expression.
country = "ES"
equity_universe = "U(IN(Equity(active,public,primary)))"
headquarters_filter = f'IN(TR.HQCountryCode,"{country}")'
universe = f"SCREEN({equity_universe},{headquarters_filter})"

print(universe)

## Step 3 · Retrieve and alphabetically sort companies

In [ ]:
# Company name, organization identifier, headquarters, and primary RIC.
fields = [
    "TR.CommonName",
    "TR.OrganizationID",
    "TR.HeadquartersCountry",
    "TR.PrimaryRICCode",
]

companies = rd.get_data(universe=universe, fields=fields)

# Identify the returned display label before sorting.
print("Available columns:", companies.columns.tolist())
name_column = "Company Common Name"

companies = (
    companies.sort_values(
        by=name_column,
        ascending=True,
        key=lambda column: column.astype("string").str.casefold(),
        na_position="last",
    )
    .reset_index(drop=True)
)
display(companies.head(20))

## Step 4 · Briefly inspect the data

In [ ]:
print("Observations:", len(companies))
print("Missing values by variable:")
display(companies.isna().sum().to_frame("Missing values"))

# Organization identifiers distinguish companies from individual equity records.
identifier_column = "Organization PermID"
print("Unique organization IDs:", companies[identifier_column].nunique())
print("Repeated organization IDs:", companies[identifier_column].dropna().duplicated().sum())

## Step 5 · Export the results

In [ ]:
output_folder = Path("output")
output_folder.mkdir(parents=True, exist_ok=True)
output_file = output_folder / "companies_spain.csv"
companies.to_csv(output_file, index=False, encoding="utf-8-sig")
print("Saved to:", output_file.resolve())

**Practice.** Change `country = "ES"` to another headquarters-country code and repeat the extraction. Match the same criteria in Screener before comparing record counts.

**Note:** Returned column labels and availability can vary by field; consult CodeCreator and adjust `name_column` / `identifier_column` if necessary.